# 數據收集與清理

## 學習目標

完成本 Notebook 後，你將能夠：

1. 說明大數據情境中常見的資料來源，例如 IoT 感測器、系統日誌、API、開放資料與企業內部系統。
2. 辨識常見資料品質問題，包括缺失值、異常值、重複值、一致性錯誤與邏輯錯誤。
3. 使用 pandas 計算缺失率、唯一值比率、重複率與異常值比例等資料品質指標。
4. 實作基本資料清理流程：去重、格式標準化、缺失值填補、異常值偵測與邏輯檢查。
5. 建立可重複執行的資料清理管線，讓後續分析或機器學習模型使用較可靠的資料。

## 情境說明

本練習以「智慧零售資料」為例。資料可能來自 POS 交易系統、會員系統、網站行為日誌與 IoT 門市感測器。這些資料在整合後，可能出現欄位缺漏、日期格式不一致、金額異常、城市名稱不一致、重複交易與不合理年齡等問題。


In [ ]:
# ── 環境設定與範例資料建立 ─────────────────────────────
# 載入本章節需要的 Python 套件，並建立一份含有常見資料品質問題的智慧零售範例資料。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from collections import Counter

# 建立一份模擬資料：混合 POS、會員與線上行為資料常見問題
raw_data = pd.DataFrame({
    "transaction_id": ["T001", "T002", "T003", "T003", "T004", "T005", "T006", "T007", "T008", "T009"],
    "customer_id": ["C01", "C02", "C03", "C03", "C04", "C05", "C06", "C07", None, "C09"],
    "city": ["臺北市", "台北市", "Taichung", "Taichung", "高雄市", "高雄", "台南市", "Tainan", "臺北市", "新北市"],
    "order_date": ["2026/01/03", "Jan-04-2026", "2026-01-05", "2026-01-05", "2026/01/06", None, "2026-13-01", "2026/01/08", "2026/01/09", "2026/01/10"],
    "amount": [1200, 899, 450, 450, -200, 9999999, np.nan, 780, 650, 1100],
    "age": [25, 34, 41, 41, 29, 999, 38, None, 17, 45],
    "source": ["POS", "API", "WebLog", "WebLog", "POS", "API", "IoT", "OpenData", "POS", "CRM"]
})

print("資料筆數與欄位數：", raw_data.shape)
print("\n範例資料：")
display(raw_data)


## 核心概念說明

在大數據處理流程中，資料收集與清理是後續分析、統計推論與 AI 模型訓練的基礎。若資料品質不佳，模型即使再複雜，也可能產生錯誤預測或偏誤結論。

### 常見資料來源

| 資料來源 | 常見格式 | 蒐集方式 | 常見挑戰 |
|---|---|---|---|
| IoT 感測器 | 時間序列、數值資料 | 串流、MQTT、Kafka | 高頻率、延遲、斷線缺漏 |
| 系統日誌 | 半結構化文字 | Logstash、Flume、NiFi | 格式不一致、資料量大 |
| API 資料 | JSON、XML | RESTful API、排程擷取 | 速率限制、欄位變動 |
| 開放資料與爬蟲 | 表格、文字、影像 | API、爬蟲框架 | 品質不一、反爬限制 |
| 企業系統 | 結構化表格 | SQL、CSV、ETL | 命名不一致、歷史資料缺漏 |

### 常見資料品質問題

1. 缺失值：欄位沒有資料，例如會員 ID 或年齡缺漏。
2. 異常值：偏離合理範圍，例如單筆消費金額過高。
3. 重複值：同一交易或同一客戶被重複記錄。
4. 一致性錯誤：格式、命名或單位不一致，例如「臺北市」與「台北市」。
5. 邏輯錯誤：格式看似正確但不符合領域規則，例如年齡為 999。


In [ ]:
# ── 示範：資料品質指標計算 ─────────────────────────────
# 這段程式碼示範如何量化資料品質，包括缺失率、唯一值比率、重複率與簡單異常值比例。

import numpy as np
import pandas as pd

raw_data = pd.DataFrame({
    "transaction_id": ["T001", "T002", "T003", "T003", "T004", "T005", "T006", "T007", "T008", "T009"],
    "customer_id": ["C01", "C02", "C03", "C03", "C04", "C05", "C06", "C07", None, "C09"],
    "city": ["臺北市", "台北市", "Taichung", "Taichung", "高雄市", "高雄", "台南市", "Tainan", "臺北市", "新北市"],
    "order_date": ["2026/01/03", "Jan-04-2026", "2026-01-05", "2026-01-05", "2026/01/06", None, "2026-13-01", "2026/01/08", "2026/01/09", "2026/01/10"],
    "amount": [1200, 899, 450, 450, -200, 9999999, np.nan, 780, 650, 1100],
    "age": [25, 34, 41, 41, 29, 999, 38, None, 17, 45]
})

quality_report = pd.DataFrame({
    "missing_rate": raw_data.isna().mean(),
    "unique_rate": raw_data.nunique(dropna=True) / len(raw_data),
    "dtype": raw_data.dtypes.astype(str)
})

duplicate_rate = raw_data.duplicated(subset=["transaction_id"]).mean()

# 使用 IQR 法偵測 amount 欄位異常值
amount = raw_data["amount"].dropna()
q1 = amount.quantile(0.25)
q3 = amount.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outlier_rate = ((amount < lower_bound) | (amount > upper_bound)).mean()

print("欄位品質報告：")
display(quality_report)
print(f"交易 ID 重複率：{duplicate_rate:.2%}")
print(f"金額異常值比例：{outlier_rate:.2%}")
print(f"IQR 合理範圍：{lower_bound:.2f} 到 {upper_bound:.2f}")


## 資料清理策略

資料清理不是單純把問題資料刪除，而是根據資料用途、業務規則與統計風險選擇適當處理方式。

### 常見處理原則

| 問題類型 | 可用方法 | 實務注意事項 |
|---|---|---|
| 缺失值 | 刪除、平均數填補、中位數填補、類別眾數填補 | 若缺失不是隨機發生，直接刪除可能造成偏差 |
| 異常值 | IQR、Z 分數、上下限規則 | 真實極端事件不一定要刪除，需搭配領域知識 |
| 重複值 | 依主鍵去重、保留最新紀錄、合併資訊 | 不同來源整合時，要確認是否真的是同一筆資料 |
| 格式不一致 | 日期標準化、文字正規化、單位轉換 | 建議建立對照表或資料字典 |
| 邏輯錯誤 | 規則檢查、範圍檢查、人工審核 | 規則應可追蹤，避免黑箱式修正 |

### 本練習採用的清理規則

1. 交易編號重複時，只保留第一筆。
2. 城市名稱統一為正式中文名稱。
3. 日期統一轉成 pandas datetime，無法解析者標記為缺失。
4. 年齡合理範圍設定為 0 到 120。
5. 金額必須大於等於 0，且用 IQR 偵測過大異常值。
6. 數值欄位缺失使用中位數填補。


In [ ]:
# ── 示範：建立資料清理管線 ─────────────────────────────
# 這段程式碼示範一個可重複執行的資料清理流程，包含去重、類別標準化、日期解析、邏輯規則檢查與缺失值填補。

import numpy as np
import pandas as pd

raw_data = pd.DataFrame({
    "transaction_id": ["T001", "T002", "T003", "T003", "T004", "T005", "T006", "T007", "T008", "T009"],
    "customer_id": ["C01", "C02", "C03", "C03", "C04", "C05", "C06", "C07", None, "C09"],
    "city": ["臺北市", "台北市", "Taichung", "Taichung", "高雄市", "高雄", "台南市", "Tainan", "臺北市", "新北市"],
    "order_date": ["2026/01/03", "Jan-04-2026", "2026-01-05", "2026-01-05", "2026/01/06", None, "2026-13-01", "2026/01/08", "2026/01/09", "2026/01/10"],
    "amount": [1200, 899, 450, 450, -200, 9999999, np.nan, 780, 650, 1100],
    "age": [25, 34, 41, 41, 29, 999, 38, None, 17, 45]
})

def clean_retail_data(df):
    cleaned = df.copy()

    # 1. 依交易編號去重
    cleaned = cleaned.drop_duplicates(subset=["transaction_id"], keep="first").reset_index(drop=True)

    # 2. 城市名稱標準化
    city_map = {
        "臺北市": "台北市",
        "台北市": "台北市",
        "Taichung": "台中市",
        "高雄": "高雄市",
        "高雄市": "高雄市",
        "Tainan": "台南市",
        "台南市": "台南市",
        "新北市": "新北市"
    }
    cleaned["city"] = cleaned["city"].map(city_map).fillna(cleaned["city"])

    # 3. 日期解析：無法解析的日期會轉為 NaT
    cleaned["order_date"] = pd.to_datetime(cleaned["order_date"], errors="coerce")

    # 4. 年齡邏輯檢查：只保留 0 到 120 歲
    cleaned.loc[~cleaned["age"].between(0, 120), "age"] = np.nan

    # 5. 金額邏輯檢查：金額不可為負數
    cleaned.loc[cleaned["amount"] < 0, "amount"] = np.nan

    # 6. 使用 IQR 法標記過大或過小的金額異常值
    valid_amount = cleaned["amount"].dropna()
    q1 = valid_amount.quantile(0.25)
    q3 = valid_amount.quantile(0.75)
    iqr = q3 - q1
    lower_bound = max(0, q1 - 1.5 * iqr)
    upper_bound = q3 + 1.5 * iqr
    amount_outlier = (cleaned["amount"] < lower_bound) | (cleaned["amount"] > upper_bound)
    cleaned.loc[amount_outlier, "amount"] = np.nan

    # 7. 數值欄位缺失使用中位數填補
    cleaned["amount"] = cleaned["amount"].fillna(cleaned["amount"].median())
    cleaned["age"] = cleaned["age"].fillna(cleaned["age"].median())

    return cleaned

cleaned_data = clean_retail_data(raw_data)

print("清理後資料筆數與欄位數：", cleaned_data.shape)
print("\n清理後資料：")
display(cleaned_data)
